# 01. Bronze Ingestion & Telemetry Profiling (42.4M Behavioral Events)

### Architectural Purpose
In this notebook, we establish our **Tier 1 (Raw Event Logs)** baseline using the public **REES46 Multi-Category Behavioral Event Dataset** (`2019-Oct.csv.gz`, ~42.45 million stateless events). 

**Why this matters for LLM & Agentic Benchmarking:**
* **No Silent Schema Inference:** Letting Spark infer schemas over multi-million-row raw logs wastes compute and risks misclassifying high-cardinality IDs or timestamps. We enforce an explicit `StructType` schema at read time.
* **Delta Lake Storage Optimization:** Converting raw gzipped CSV logs (~1.4 GB compressed / ~5.67 GB uncompressed) into an uncompressed **Delta table** (`workspace.default.bronze_events`) enables Parquet columnar compression (~1.2 GB on disk) and transaction-log min/max data skipping without introducing small-file partitioning overhead.
* **Baseline Anomaly Audit:** Before building our Tier 2 (Semantic Layer) and Tier 3 (Ontology) abstractions, we profile null rates and funnel distributions to document the exact data-quality traps that cause naive Text-to-SQL agents to hallucinate.  


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as F

raw_path = "/Volumes/workspace/default/raw_events/2019-Oct.csv.gz"

event_schema = StructType([
    StructField("event_time", StringType(), True), 
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

df_raw = (
    spark.read
    .option("header", "true")
    .schema(event_schema)
    .csv(raw_path)
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .drop("event_time")
)

display(df_raw.limit(100))


In [0]:
(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_events")
)

print("Saved bronze_events successfully!")


In [0]:
bronze = spark.table("workspace.default.bronze_events")

print(f"Total raw events: {bronze.count():,}")

# Breakdown by event_type
display(
    bronze.groupBy("event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("user_id").alias("distinct_users"),
        F.countDistinct("user_session").alias("distinct_sessions")
    )
    .orderBy(F.desc("event_count"))
)

# Check null rates (especially category_code, brand, and user_session)
null_summary = bronze.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 2).alias(f"{c}_null_pct")
    for c in bronze.columns
])
display(null_summary)
